In [1]:
from pathlib import Path
import pandas as pd
import re

RAW_BASE = Path("/Users/saifali03/Desktop/SML/project/data/raw")
REGISTRY_OUT = Path("/Users/saifali03/Desktop/SML/project/data/interim/session_registry.csv")

pattern = re.compile(r"^(?P<machine_id>.+)_(?P<date>\d{8})_(?P<run_number>\d{3})\.csv$")

rows = []

for csv_path in sorted(RAW_BASE.rglob("*.csv")):
    m = pattern.match(csv_path.name)
    if not m:
        continue

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Skipping {csv_path}: {e}")
        continue

    if "timestamp_utc" not in df.columns or df.empty:
        print(f"Skipping {csv_path}: missing timestamp_utc or empty")
        continue

    ts = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce").dropna()
    if ts.empty:
        print(f"Skipping {csv_path}: timestamps could not be parsed")
        continue

    start_utc = ts.min()
    end_utc = ts.max()
    duration_seconds = (end_utc - start_utc).total_seconds()
    n_rows = len(df)

    if len(ts) >= 2:
        interval_seconds = ts.sort_values().diff().dt.total_seconds().median()
    else:
        interval_seconds = None

    machine_id = m.group("machine_id")
    date_str = pd.to_datetime(m.group("date"), format="%Y%m%d").strftime("%Y-%m-%d")
    run_number = m.group("run_number")
    session_id = f"{machine_id}_{m.group('date')}_{run_number}"

    os_family = df["os_family"].dropna().iloc[0] if "os_family" in df.columns and df["os_family"].notna().any() else ""

    rows.append({
        "session_id": session_id,
        "machine_id": machine_id,
        "date": date_str,
        "run_number": run_number,
        "file_path": str(csv_path.resolve()),
        "file_name": csv_path.name,
        "start_utc": start_utc.isoformat(),
        "end_utc": end_utc.isoformat(),
        "duration_seconds": round(duration_seconds, 3),
        "n_rows": n_rows,
        "interval_seconds": interval_seconds,
        "logger_version": "reconstructed_from_raw_files",
        "os_family": os_family,
        "notes": "NA"
    })

registry = pd.DataFrame(rows).sort_values(["machine_id", "date", "run_number"])
REGISTRY_OUT.parent.mkdir(parents=True, exist_ok=True)
registry.to_csv(REGISTRY_OUT, index=False)

print(f"Rebuilt registry with {len(registry)} sessions at: {REGISTRY_OUT}")

Rebuilt registry with 2 sessions at: /Users/saifali03/Desktop/SML/project/data/interim/session_registry.csv
